In [1]:
!pip install neo4j


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 7.4 MB/s eta 0:00:00


In [25]:
import json
import os
from neo4j import GraphDatabase

# 1. Setup Connection (Replace with your Aura details or local host)
# If using Aura, the URI looks like: "neo4j+s://xxxxxxxx.databases.neo4j.io"
URI = os.getenv('NEO4J_URI')
AUTH = (os.getenv('NEO4J_USERNAME'), os.getenv('NEO4J_PASSWORD'))


In [22]:
import json
import os
from neo4j import GraphDatabase

# 1. Setup Connection
URI = os.getenv('NEO4J_URI')
# The default user for Aura is almost always 'neo4j'
USER = os.getenv('NEO4J_USERNAME')
PASSWORD = os.getenv('NEO4J_PASSWORD') # Check your downloaded credentials.txt

def run_query():
    try:
        # The driver handles the encryption and handshake via the URI
        with GraphDatabase.driver(URI, auth=(USER, PASSWORD)) as driver:
            driver.verify_connectivity()
            print("Connection successful!")

            # Example: simple query to test
            # with driver.session() as session:
            #     result = session.run("RETURN 'Hello Graph' AS message")
            #     print(result.single()["message"])

    except Exception as e:
        print(f"Failed to connect: {e}")

if __name__ == "__main__":
    run_query()

Connection successful!


In [26]:
driver = GraphDatabase.driver(URI, auth=AUTH)

In [4]:
def load_graph_json(path):

    with open(path, "r") as f:
        data = json.load(f)

    return data

In [27]:
input_path = "/content/drive/MyDrive/mental-health-research/DEPRESSION/mhgap_decision_graph.json"

In [ ]:
input_path = "/content/drive/MyDrive/mental-health-research/SELF-HARM SUICIDE/mhgap_decision_graph.json"

In [ ]:
input_path ="/content/drive/MyDrive/mental-health-research/DISORDERS DUE TO SUBSTANCE USE/mhgap_decision_graph.json"

In [ ]:
input_path ="/content/drive/MyDrive/mental-health-research/OTHER SIGNIFICANT MENTAL HEALTH COMPLAINTS/mhgap_decision_graph.json"

In [8]:
data = load_graph_json(input_path)


In [28]:
def create_constraints(tx):

    tx.run("""
    CREATE CONSTRAINT step_id_unique IF NOT EXISTS
    FOR (s:Step)
    REQUIRE s.id IS UNIQUE
    """)
with driver.session() as session:
    session.execute_write(create_constraints)

# Now every Step node must have a unique id.

In [29]:
def insert_node(tx, node, module):
    # Add the module to the node dictionary so it's saved with the node
    node_data = dict(node)
    node_data['module'] = module

    tx.run(
        """
        MERGE (s:Step {id: $id})
        SET s += $properties
        """,
        id=node["id"],
        properties=node_data
    )

In [30]:
def insert_relationship(tx, rel):

    tx.run(
        """
        MATCH (a:Step {id: $from_id})
        MATCH (b:Step {id: $to_id})
        MERGE (a)-[r:NEXT {condition:$condition}]->(b)
        """,
        from_id=rel["from"],
        to_id=rel["to"],
        condition=rel["condition"]
    )

In [31]:
def import_graph(json_data):

    module_id = json_data["module"]["id"]

    with driver.session() as session:

        # insert nodes
        for node in json_data["nodes"]:
            session.execute_write(insert_node, node, module_id)

        # insert relationships
        for rel in json_data["relationships"]:
            session.execute_write(insert_relationship, rel)

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [32]:
data = load_graph_json(input_path)

import_graph(data)
